# CertVIC -- LLaVA-OneVision-7B eval on Kaggle (SELF-DOWNLOAD, no weights input)

Third open VLM for the multi-model pilot. Downloads `llava-hf/llava-onevision-qwen2-7b-ov-hf`
at runtime (Internet ON), loads it in **bf16 sharded across 2x T4** (no bitsandbytes), and runs
CertVIC eval via the in-process `certvic.eval.run_eval` (leakage / evidence / resume / manifest
gates intact, `provider_name=llava_onevision_7b`).

**Attach only 3 inputs:** CertVIC code, presence data, absent-object control data.
**No model-weights dataset.** Outputs: `pred_llava_onevision_7b_{presence,control}_merged.jsonl`
(+ `llava_preds.zip`). Pilot-only; no paper claims.

> LLaVA-OneVision is a NATIVE transformers model (no remote code), so it avoids InternVL's
> `all_tied_weights_keys` break -- we pin `transformers==4.49.0` (has `LlavaOnevisionForConditionalGeneration`,
> pre-4.50 churn). Like the InternVL notebook we **avoid bitsandbytes** (broken on the
> Py3.12/CUDA-12.8 image) and load bf16 across both T4s; single GPU falls back to 4-bit.

## Settings & inputs

- **Accelerator: GPU T4 x2 (recommended).** LLaVA-OneVision-7B bf16 is ~16 GB -- fits sharded
  across two T4s, not one. Single T4 falls back to 4-bit. **Internet: ON** (model download).
- **Attach 3 datasets** (auto-detected): CertVIC code (has `certvic/`), presence
  (`certvic_main200_session2_data.zip`, 91 tasks), absent-object control
  (`certvic_absent_object_control.zip`, 120 tasks).
- Optional `HF_TOKEN` Kaggle Secret (not required; the model is public).
- Run top-to-bottom; **stop at the smoke test** and confirm yes/no first.

In [ ]:
# CELL 1 -- pin a transformers with LLaVA-OneVision BEFORE importing it. (No bitsandbytes:
# the bf16 multi-GPU path needs none; the 1-GPU 4-bit fallback installs a current one on demand.)
import sys, subprocess, os
PINS = ["transformers==4.49.0", "accelerate>=0.34", "sentencepiece", "huggingface_hub>=0.24"]
_SENTINEL = "/kaggle/working/.deps_installed_llava"
if not os.path.exists(_SENTINEL):
    print("installing pinned stack (one-time)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    open(_SENTINEL, "w").write("ok")
    print("done.")
else:
    print("deps already installed (sentinel present).")
if "transformers" in sys.modules and not getattr(sys.modules["transformers"], "__version__", "").startswith("4.49"):
    raise SystemExit("Stale transformers already imported -> Run menu > 'Restart & Run All' once.")

In [ ]:
# CELL 2 -- verify versions + that LLaVA-OneVision is available.
import torch, transformers
from transformers import LlavaOnevisionForConditionalGeneration  # noqa: F401  (import = availability check)
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| CUDA", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count())
assert transformers.__version__.startswith("4.49"), "need transformers 4.49.x -> Run > 'Restart & Run All'."
print("version stack OK; LlavaOnevisionForConditionalGeneration importable")

In [ ]:
# CELL 3 -- auto-detect Kaggle inputs (override only if auto-detect fails).
import glob, os, json

CERTVIC_DIR    = None   # parent dir that contains 'certvic/'
PRESENCE_INPUT = None   # dir with the 91-task pilot_eval_tasks_reviewed.jsonl
CONTROL_INPUT  = None   # dir with the 120-task pilot_eval_tasks_reviewed.jsonl

def _find_certvic_parent():
    for p in glob.glob("/kaggle/input/**/certvic", recursive=True):
        if os.path.isdir(p) and os.path.exists(os.path.join(p, "eval", "run_eval.py")):
            return os.path.dirname(p)
    return None

def _find_task_bundles():
    pres = ctrl = None
    for p in glob.glob("/kaggle/input/**/pilot_eval_tasks_reviewed.jsonl", recursive=True):
        try:
            n = sum(1 for _ in open(p))
        except OSError:
            continue
        if n == 91 and pres is None:
            pres = os.path.dirname(p)
        elif n == 120 and ctrl is None:
            ctrl = os.path.dirname(p)
    return pres, ctrl

CERTVIC_DIR = CERTVIC_DIR or _find_certvic_parent()
_p, _c = _find_task_bundles()
PRESENCE_INPUT = PRESENCE_INPUT or _p
CONTROL_INPUT  = CONTROL_INPUT or _c
MODEL_DIR = "/kaggle/working/hf_models/llava-ov-7b"

print("CERTVIC_DIR   :", CERTVIC_DIR)
print("PRESENCE_INPUT:", PRESENCE_INPUT, "(91-task bundle)")
print("CONTROL_INPUT :", CONTROL_INPUT, "(120-task bundle)")
print("MODEL_DIR     :", MODEL_DIR, "(downloaded at runtime)")
_missing = [n for n, v in [("CERTVIC code", CERTVIC_DIR), ("presence data", PRESENCE_INPUT),
                           ("absent-object control data", CONTROL_INPUT)] if not v]
if _missing:
    raise RuntimeError("Missing required input(s): " + ", ".join(_missing) + ". Attach the 3 datasets.")
sys.path.insert(0, CERTVIC_DIR)
import certvic; print("certvic OK from", os.path.dirname(certvic.__file__))

In [ ]:
# CELL 4 -- download LLaVA-OneVision at runtime (cached + resumable). Internet ON.
import os
from huggingface_hub import snapshot_download
REPO_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"

def _hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

os.makedirs(MODEL_DIR, exist_ok=True)
if os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("reusing cached weights at", MODEL_DIR)
else:
    print("downloading", REPO_ID, "-> ", MODEL_DIR, "(~16 GB, one time)...")
    snapshot_download(repo_id=REPO_ID, local_dir=MODEL_DIR, token=_hf_token())
    print("download complete.")
assert os.path.exists(os.path.join(MODEL_DIR, "config.json")), "snapshot missing config.json"

In [ ]:
# CELL 5 -- CertVIC run config (label only; weights load from MODEL_DIR in the patch cell).
WORK = "/kaggle/working"
CFG = f"{WORK}/kaggle_llava.yaml"
cfg_lines = [
    "mode: kaggle_open_vlm", "provider: llava_onevision_7b",
    "model_id: llava-hf/llava-onevision-qwen2-7b-ov-hf",
    "device: cuda", "dtype: bfloat16", "batch_size: 1",
    "max_new_tokens: 16", "temperature: 0.0", "paid_services_enabled: false",
]
open(CFG, "w").write("\n".join(cfg_lines) + "\n")
print("wrote", CFG)

In [ ]:
# CELL 6 -- load LLaVA-OneVision (bf16 sharded on >=2 GPUs; 4-bit on 1 GPU) + patch the scaffold.
import sys, subprocess, time as _time
import torch
from PIL import Image
from transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor
import certvic.providers.open_vlm as ovlm

transformers.logging.set_verbosity_error()
processor = AutoProcessor.from_pretrained(MODEL_DIR)
_ngpu = torch.cuda.device_count()
_common = dict(torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
if _ngpu >= 2:
    print(f"{_ngpu} GPUs -> bf16 sharded (device_map=auto, no bitsandbytes).")
    model = LlavaOnevisionForConditionalGeneration.from_pretrained(MODEL_DIR, device_map="auto", **_common).eval()
else:
    print("Only 1 GPU -> bf16 (~16 GB) will not fit a 16 GB T4; trying 4-bit. Prefer GPU T4 x2.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes>=0.45.0"], check=True)
    from transformers import BitsAndBytesConfig
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = LlavaOnevisionForConditionalGeneration.from_pretrained(
        MODEL_DIR, quantization_config=qcfg, device_map={"": 0}, **_common).eval()
GEN = dict(max_new_tokens=16, do_sample=False,
           pad_token_id=processor.tokenizer.eos_token_id)   # deterministic; pad=eos silences the warning

_PROG = {"n": 0, "tag": "", "t0": None}
@torch.inference_mode()
def _vlm_answer(self, image_path, prompt):
    if _PROG["t0"] is None:
        _PROG["t0"] = _time.time()
    image = Image.open(image_path).convert("RGB")
    image.thumbnail((384, 384))  # cap longest side -> LLaVA-OneVision uses ~1 base tile -> fast
    conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=text, return_tensors="pt").to(model.device, torch.bfloat16)
    out = model.generate(**inputs, **GEN)
    ans = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    _PROG["n"] += 1
    if _PROG["n"] <= 2 or _PROG["n"] % 20 == 0:
        rate = _PROG["n"] / max(_time.time() - _PROG["t0"], 1e-6)
        print(f"  {_PROG['tag']}: {_PROG['n']} gens | {rate:.2f} gen/s", flush=True)
    return ans

ovlm.OpenVLMProvider.load = lambda self: None
ovlm.OpenVLMProvider.answer = _vlm_answer
print("LLaVA-OneVision loaded + OpenVLMProvider.answer patched; GPUs:", _ngpu)

In [ ]:
# CELL 7 -- SMOKE TEST: 2 presence + 2 control examples must parse to yes/no.
import json
from certvic.eval.parse import parse_answer

def smoke(job_input, label, k=2):
    rows = [json.loads(l) for l in open(f"{job_input}/pilot_eval_tasks_reviewed.jsonl")][:k]
    ok = True
    for r in rows:
        imgs = {"original": f"{job_input}/orig/{os.path.basename(r['original_image_path'])}",
                "edited":   f"{job_input}/{os.path.basename(r['edited_image_path'])}"}
        for variant, img in imgs.items():
            raw = _vlm_answer(None, img, r["question_original"])
            p = parse_answer(raw, "yes_no", strict=True)
            print(f"[{label}] {r['item_id'][:20]:20s} {variant:8s} raw={raw!r:12s} "
                  f"parsed={p.parsed_answer} ok={p.parse_ok}")
            ok = ok and p.parse_ok and p.parsed_answer in ("yes", "no")
    return ok

_ok = smoke(PRESENCE_INPUT, "presence") and smoke(CONTROL_INPUT, "control")
if not _ok:
    raise RuntimeError("Smoke test produced non-yes/no answers -- fix the chat template/dtype before the full run.")
print("SMOKE OK -- answers parse to yes/no. Proceed.")

In [ ]:
# CELL 8 -- remap bundle image paths, project to the strict TaskItem schema, then run CertVIC eval.
import json, time
from certvic.eval.run_eval import run_eval
from certvic.schema import TaskItem
from certvic.schema.edit import EditSpec
from certvic.schema.source import SourceImageRecord

def _to_taskitem(r):
    # Project the flat reviewed/preview schema onto the nested TaskItem run_eval requires.
    src = SourceImageRecord(source_id=r['source_id'], source_name='ADE20K', license_category='pointer_only')
    ed = EditSpec(edit_id=r['edit_id'], source_id=r['source_id'], edit_type=r['edit_type'],
                  task_family=r['task_family'], domain=r['domain'], expected_effect=r['expected_effect'])
    m = dict(r.get('metadata') or {})
    m.setdefault('evidence_status', r.get('evidence_status', 'HUMAN_REVIEWED_NON_EVIDENCE'))
    return TaskItem(item_id=r['item_id'], source=src, edit=ed,
                    original_image_path=r['original_image_path'], edited_image_path=r['edited_image_path'],
                    question_original=r['question_original'], question_edited=r['question_edited'],
                    answer_original=r['answer_original'], answer_edited=r['answer_edited'],
                    required_change=r['required_change'], answer_format=r['answer_format'],
                    task_family=r['task_family'], domain=r['domain'], split=r['split'], metadata=m)

def remap_tasks(job_input, name):
    rows = [json.loads(l) for l in open(f'{job_input}/pilot_eval_tasks_reviewed.jsonl')]
    miss = 0
    for r in rows:
        ob, eb = os.path.basename(r['original_image_path']), os.path.basename(r['edited_image_path'])
        r['original_image_path'] = f'{job_input}/orig/{ob}'
        r['edited_image_path']   = f'{job_input}/{eb}'
        miss += (not os.path.exists(r['original_image_path'])) + (not os.path.exists(r['edited_image_path']))
    assert miss == 0, f'{name}: {miss} images not found -- check bundle layout/path.'
    needs_convert = bool(rows) and 'source' not in rows[0]   # presence=flat->convert; control=already nested
    if needs_convert:
        rows = [json.loads(_to_taskitem(r).model_dump_json()) for r in rows]
    dst = f'{WORK}/tasks_{name}.jsonl'
    open(dst, 'w').writelines(json.dumps(r) + '\n' for r in rows)
    print(f'{name}: {len(rows)} tasks -> {dst} | schema: ' + ('flat->TaskItem' if needs_convert else 'already nested'))
    return dst

JOBS = [
    {'name': 'presence', 'input': PRESENCE_INPUT, 'run_id': 'main200_llava_onevision_7b_presence',
     'out': 'pred_llava_onevision_7b_presence_merged.jsonl'},
    {'name': 'control', 'input': CONTROL_INPUT, 'run_id': 'main200_llava_onevision_7b_control',
     'out': 'pred_llava_onevision_7b_control_merged.jsonl'},
]
for job in JOBS:
    tasks = remap_tasks(job['input'], job['name'])
    out = f"{WORK}/{job['out']}"
    _PROG['n'], _PROG['tag'], _PROG['t0'] = 0, job['name'], None
    print(f"running {job['name']} ...", flush=True)
    t0 = time.time()
    summary = run_eval(config_path=CFG, tasks_path=tasks, out_path=out,
                       provider_name='llava_onevision_7b', run_id=job['run_id'],
                       num_shards=1, strict_leakage=True, evidence_run=True,
                       fail_fast=False, overwrite=False)
    print(job['name'], summary, f"({time.time()-t0:.0f}s) -> {out}")

In [ ]:
# CELL 9 -- final parse rate + yes/no distribution + provider stamp.
import collections, json
for job in JOBS:
    rows = [json.loads(l) for l in open(f"{WORK}/{job['out']}")]
    ans = collections.Counter(r["parsed_answer"] for r in rows)
    okr = sum(r["parse_ok"] for r in rows) / len(rows)
    print(f"{job['name']:9s}: {len(rows):3d} preds | parse_ok={okr:.3f} | answers={dict(ans)} "
          f"| provider={sorted({r['provider_name'] for r in rows})}")

In [ ]:
# CELL 10 -- zip predictions + manifests for download.
import glob, zipfile
files = sorted(glob.glob(f"{WORK}/pred_llava_onevision_7b_*_merged.jsonl")
               + glob.glob(f"{WORK}/pred_llava_onevision_7b_*_merged.jsonl.run_manifest.json"))
with zipfile.ZipFile(f"{WORK}/llava_preds.zip", "w") as z:
    for f in files:
        z.write(f, os.path.basename(f))
print("wrote llava_preds.zip ->", [os.path.basename(f) for f in files])

## Back on the Mac

Download `pred_llava_onevision_7b_presence_merged.jsonl` and
`pred_llava_onevision_7b_control_merged.jsonl` (or `llava_preds.zip`), then:

```bash
cd /path/to/certVIC
python3 scripts/pilot_report_from_raw.py \
  --provider llava_onevision_7b --model-name llava-hf/llava-onevision-qwen2-7b-ov-hf \
  --run-label llava_onevision_7b \
  --raw-presence /path/to/pred_llava_onevision_7b_presence_merged.jsonl \
  --raw-control  /path/to/pred_llava_onevision_7b_control_merged.jsonl
```

Writes `data/results/main_real_200/pilot_report__llava_onevision_7b/` (+ sha256-locked raw) and
completes `multimodel_pilot_summary.{md,csv,json}` (3/3 models). REFUSES if a file is missing or
its `provider_name` != llava_onevision_7b. Pilot-only; no paper-grade claim.